# `vkpandas` Complex DataFrame Demo (Vulkan GPU)

End-to-end walkthrough of every `vkpandas` `Series`, `DataFrame`, `GroupBy`, `Rolling`, `merge`, and I/O operation running on top of `vknumpy` (`llama.cpp` `ggml-vulkan` + `Tencent/ncnn`).

In [1]:
import sys
from pathlib import Path
_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import time
from pathlib import Path
import numpy as np
import vknumpy as vnp
import vkpandas as vpd

vpd.Series([str(vnp.device_info()["name"]), str(vnp.device_info()["backend"])], name="device").show()

0	AMD Custom GPU 0405 (RADV VANGOGH)
1	ggml-vulkan
Name: device, dtype: string


ggml_vulkan: Found 1 Vulkan devices:
ggml_vulkan: 0 = AMD Custom GPU 0405 (RADV VANGOGH) (radv) | uma: 1 | rebar: 1 | fp16: 0 | bf16: 0 | fp4: 0 | warp size: 32 | shared memory: 65536 | int dot: 1 | matrix cores: none


## 1. Create a Complex Multi-Column DataFrame (50,000 Orders + Customer Lookup Table)

In [2]:
rng = np.random.default_rng(42)
N = 50_000

regions = rng.choice(["north", "south", "east", "west", "central"], size=N)
categories = rng.choice(["electronics", "apparel", "home", "sports", "books"], size=N)
customer_id = rng.integers(1000, 3500, size=N, dtype=np.int32)
unit_price = rng.uniform(5.0, 500.0, size=N).astype(np.float32)
qty = rng.integers(1, 10, size=N, dtype=np.int32)
discount = rng.uniform(0.0, 0.30, size=N).astype(np.float32)

# Inject ~5% missing values (NaN) into discount and unit_price
discount[rng.random(N) < 0.05] = np.nan
unit_price[rng.random(N) < 0.02] = np.nan

orders = vpd.DataFrame({
    "order_id": np.arange(N, dtype=np.int32),
    "customer_id": customer_id,
    "region": regions,
    "category": categories,
    "unit_price": unit_price,
    "qty": qty,
    "discount": discount,
})

# Secondary customer dimension table (2,000 unique customers for inner join)
cust_ids = np.arange(1000, 3000, dtype=np.int32)
customers = vpd.DataFrame({
    "customer_id": cust_ids,
    "tier": rng.choice(["bronze", "silver", "gold", "platinum"], size=len(cust_ids)),
    "loyalty_score": rng.uniform(0.0, 100.0, size=len(cust_ids)).astype(np.float32),
})

orders.show(5)
customers.show(5)

    order_id  customer_id  region     category          unit_price  qty             discount
0          0         2838   north  electronics   495.0417785644531    7  0.25301915407180786
1          1         3205    west        books  151.66746520996094    4  0.22623266279697418
2          2         2429    west         home   456.1916809082031    7  0.15036477148532867
3          3         2035    east        books  449.88934326171875    6  0.18024194240570068
4          4         1011    east         home   428.2514953613281    2   0.2201225906610489
... (50000 rows x 7 columns)
    customer_id      tier       loyalty_score
0          1000  platinum   57.29983139038086
1          1001    silver  23.116710662841797
2          1002    bronze   68.26099395751953
3          1003    silver   58.43830490112305
4          1004  platinum  60.438114166259766
... (2000 rows x 3 columns)


## 2. Inspect DataFrame (`head`, `tail`, `show`)

In [3]:
orders.head(5).show()
orders.tail(3).show()

    order_id  customer_id  region     category          unit_price  qty             discount
0          0         2838   north  electronics   495.0417785644531    7  0.25301915407180786
1          1         3205    west        books  151.66746520996094    4  0.22623266279697418
2          2         2429    west         home   456.1916809082031    7  0.15036477148532867
3          3         2035    east        books  449.88934326171875    6  0.18024194240570068
4          4         1011    east         home   428.2514953613281    2   0.2201225906610489
    order_id  customer_id  region     category          unit_price  qty              discount
0      49997         1726   north         home   51.18564987182617    5                   nan
1      49998         2507   south  electronics   82.92969512939453    3   0.03285735100507736
2      49999         1767    west        books  53.257808685302734    8  0.012818915769457817


## 3. Missing-Value Handling (`isna`, `notna`, `count`, `fillna`, `ffill`, `bfill`, `dropna`)

In [4]:
orders["discount_clean"] = orders["discount"].fillna(0.0)
orders["price_ffill"] = orders["unit_price"].ffill()

clean_orders = orders.dropna(subset=["unit_price"])

vpd.DataFrame({
    "metric": ["total_rows", "non_null_discount", "missing_discount", "after_dropna"],
    "count": [
        len(orders),
        orders["discount"].count(),
        int(vnp.count_nonzero(orders["discount"].isna().values).item()),
        len(clean_orders),
    ],
}).show()

orders["discount"].head(6).bfill().show()

               metric  count
0          total_rows  50000
1   non_null_discount  47530
2    missing_discount   2470
3        after_dropna  49049
0	0.25301915407180786
1	0.22623266279697418
2	0.15036477148532867
3	0.18024194240570068
4	0.2201225906610489
5	0.08761393278837204
Name: discount, dtype: float32


## 4. Computed Columns & GPU `Series` Aggregations (`sum`, `mean`, `std`, `var`, `min`, `max`)

In [5]:
clean_orders["gross_sales"] = clean_orders["unit_price"] * clean_orders["qty"].astype(np.float32)
clean_orders["net_sales"] = clean_orders["gross_sales"] * (1.0 - clean_orders["discount_clean"])

ns = clean_orders["net_sales"]
vpd.DataFrame({
    "stat": ["sum", "mean", "std", "var", "min", "max"],
    "net_sales": [ns.sum(), ns.mean(), ns.std(), ns.var(), ns.min(), ns.max()],
}).show()

[0 AMD Custom GPU 0405 (RADV VANGOGH)]  queueC=1[4]  queueT=0[1]  prefer-slm=0  rebar=1  r-score=11
[0 AMD Custom GPU 0405 (RADV VANGOGH)]  fp16-p/s/u/a=1/1/1/1  int8-p/s/u/a=1/1/1/1  bf16-p/s=1/0
[0 AMD Custom GPU 0405 (RADV VANGOGH)]  subgroup=64(32~64)  ops=1/1/1/1/1/1/1/1/1/1
[0 AMD Custom GPU 0405 (RADV VANGOGH)]  fp16-cm=0  int8-cm=0  bf16-cm=0  fp8-cm=0  fp6-cm=0  fp4-cm=0


    stat          net_sales
0    sum         53141972.0
1   mean  1083.446533203125
2    std  901.9020385742188
3    var        813427.3125
4    min  3.775254964828491
5    max   4478.45458984375


## 5. Categorical / String Analysis (`unique`, `nunique`, `value_counts`, `isin`)

In [6]:
clean_orders["region"].unique().show()

vc = clean_orders["category"].value_counts()
vpd.DataFrame({
    "category": list(vc.keys()),
    "count": list(vc.values()),
}).show()

focus = clean_orders[clean_orders["category"].isin(["electronics", "sports"])]
focus[["order_id", "region", "category", "net_sales"]].show(5)

0	north
1	west
2	east
3	central
4	south
Name: region, dtype: string
       category  count
0        sports   9965
1       apparel   9844
2         books   9765
3   electronics   9753
4          home   9722
    order_id  region     category           net_sales
0          0   north  electronics   2588.507080078125
1         13    west  electronics   62.29144287109375
2         14    west       sports    2425.20654296875
3         16    east  electronics    1070.77783203125
4         17   north       sports  1007.9359741210938
... (19718 rows x 4 columns)


## 6. GPU Boolean Mask Filtering & Column Selection / `drop`

In [7]:
high_value = clean_orders[
    (clean_orders["net_sales"] > 2000.0) &
    ((clean_orders["region"] == "north") | (clean_orders["region"] == "west"))
]

trimmed = high_value.drop(["discount", "price_ffill"])
trimmed[["order_id", "region", "category", "net_sales"]].show(5)

    order_id  region     category          net_sales
0          0   north  electronics  2588.507080078125
1          2    west         home      2713.17578125
2         14    west       sports   2425.20654296875
3         32   north      apparel  3011.517822265625
4         42    west         home  2414.603759765625
... (3293 rows x 4 columns)


## 7. Single-Column & Multi-Column GPU Sorting (`sort_values`)

In [8]:
trimmed.sort_values("net_sales", ascending=False)[["order_id", "region", "category", "net_sales"]].show(5)
trimmed.sort_values(["region", "category", "net_sales"], ascending=True)[["order_id", "region", "category", "net_sales"]].show(6)

    order_id  region     category         net_sales
0       8072    west        books  4478.45458984375
1      27086   north      apparel    4454.486328125
2      21843    west  electronics   4436.2236328125
3      49819   north       sports  4406.14599609375
4      34314   north  electronics  4403.00244140625
... (3293 rows x 4 columns)
    order_id  region     category           net_sales
0       6563   north  electronics    2004.58544921875
1      20437   north  electronics  2008.4442138671875
2      32659   north  electronics  2014.1756591796875
3      22655   north  electronics   2014.529052734375
4      37043   north  electronics  2017.7974853515625
5      30639   north  electronics  2021.1112060546875
... (3293 rows x 4 columns)


## 8. GPU `groupby` Aggregations (`sum`, `mean`, `count`, `std`, `agg`)

In [9]:
by_region = clean_orders.groupby("region").agg({
    "net_sales": "sum",
    "unit_price": "mean",
    "qty": "count",
})
by_region.sort_values("net_sales", ascending=False).show()

clean_orders[["category", "unit_price", "net_sales"]].groupby("category").std().show()

     region   net_sales          unit_price   qty
0      west  10824950.0  253.55606079101562  9919
1   central  10629152.0  253.09359741210938  9712
2     south  10617768.0  253.00389099121094  9781
3      east  10575282.0     252.06689453125  9806
4     north  10494816.0  251.32321166992188  9831
       category          unit_price          net_sales
0   electronics  143.14572143554688   906.745361328125
1         books   142.8643341064453  894.9529418945312
2          home   142.0577392578125   914.973388671875
3       apparel  142.69854736328125  890.2453002929688
4        sports  144.09727478027344  902.2003784179688


## 9. GPU Inner Join (`merge`)

In [10]:
enriched = clean_orders.merge(customers, on="customer_id", how="inner")
enriched[["order_id", "customer_id", "tier", "loyalty_score", "net_sales"]].show(6)

enriched.groupby("tier").agg({"net_sales": "sum", "loyalty_score": "mean"}).sort_values("net_sales", ascending=False).show()

    order_id  customer_id      tier       loyalty_score           net_sales
0          0         2838  platinum    8.49921703338623   2588.507080078125
1          2         2429    bronze   3.732109308242798       2713.17578125
2          3         2035  platinum   86.21299743652344   2212.802490234375
3          4         1011  platinum   94.75801849365234   667.9673461914062
4          5         2531      gold  22.578636169433594  380.86370849609375
5          7         1352      gold   88.17503356933594        1098.1171875
... (39177 rows x 5 columns)


        tier   net_sales       loyalty_score
0     bronze  11802274.0  49.379573822021484
1       gold  10609622.0    48.5423583984375
2     silver  10008038.0  47.706809997558594
3   platinum  10002034.0    51.2291145324707


## 10. Time-Series & Rolling Window Ops (`cumsum`, `diff`, `shift`, `rolling`)

In [11]:
sample = enriched.head(10).copy()
sample["cum_sales"] = sample["net_sales"].cumsum()
sample["prev_sales"] = sample["net_sales"].shift(1)
sample["sales_diff"] = sample["net_sales"].diff()
sample["roll3_sum"] = sample["net_sales"].rolling(3).sum()
sample["roll3_mean"] = sample["net_sales"].rolling(3).mean()

sample[["order_id", "net_sales", "cum_sales", "sales_diff", "roll3_sum", "roll3_mean"]].show(10)

    order_id           net_sales          cum_sales           sales_diff        roll3_sum          roll3_mean
0          0   2588.507080078125  2588.507080078125                  nan              nan                 nan
1          2       2713.17578125    5301.6826171875     124.668701171875              nan                 nan
2          3   2212.802490234375    7514.4853515625    -500.373291015625  7514.4853515625    2504.82861328125
3          4   667.9673461914062    8182.4521484375   -1544.835205078125     5593.9453125        1864.6484375
4          5  380.86370849609375      8563.31640625   -287.1036376953125  3261.6337890625  1087.2113037109375
5          7        1098.1171875      9661.43359375    717.2534790039062  2146.9482421875      715.6494140625
6          8  1902.8258056640625    11564.259765625    804.7086181640625  3381.8076171875   1127.269287109375
7          9     646.23583984375    12210.494140625  -1256.5899658203125   3647.177734375  1215.7259521484375
8         

## 11. Vertical `concat` & CSV I/O (`read_csv`)

In [12]:
part_a = enriched[["order_id", "region", "tier", "net_sales"]].head(3)
part_b = enriched[["order_id", "region", "tier", "net_sales"]].tail(3)
stacked = vpd.concat([part_a, part_b])
stacked.show()

csv_path = Path("/tmp/vkpandas_demo_sample.csv")
lines = [",".join(stacked.columns)]
d = stacked.to_dict()
for i in range(len(stacked)):
    lines.append(",".join(str(d[c][i]) for c in stacked.columns))
csv_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

vpd.read_csv(csv_path).show()

    order_id  region      tier           net_sales
0          0   north  platinum   2588.507080078125
1          2    west    bronze       2713.17578125
2          3    east  platinum   2212.802490234375
3      49997   north      gold  255.92825317382812
4      49998   south    bronze   240.6145477294922
5      49999    west  platinum   420.6007995605469
    order_id  region      tier           net_sales
0          0   north  platinum   2588.507080078125
1          2    west    bronze       2713.17578125
2          3    east  platinum   2212.802490234375
3      49997   north      gold  255.92825317382812
4      49998   south    bronze   240.6145477294922
5      49999    west  platinum   420.6007995605469


## 12. Zero-Copy Interop: `vkpandas` Columns → `vknumpy` Vulkan GEMM

In [13]:
X = vnp.stack([
    enriched["unit_price"].values,
    enriched["qty"].astype(np.float32).values,
    enriched["discount_clean"].values,
    enriched["loyalty_score"].values,
], axis=1)

W = vnp.array(rng.standard_normal((3, 4)).astype(np.float32))
scores = X @ W.T  # Zero-copy transposed GEMM on Vulkan GPU!

scores.head(5).show()
scores.mean(axis=0).show()

[[ 801.58734  -199.38615   187.17534 ]
 [ 740.47     -182.69783   172.43562 ]
 [ 694.63293  -210.46622   167.52052 ]
 [ 654.45184  -201.9622    159.91966 ]
 [ 127.719025  -44.79367    30.024094]]
(shape=(5, 3), dtype=float32)
[ 389.90857  -119.713684   93.57711 ]
(shape=(3,), dtype=float32)
